In [1]:
import sys
import os

PROJECT_ROOT = r"C:\Users\Sahil.s.Vernekar\OneDrive\Documents\ML\Churn\ml_system"
sys.path.insert(0, PROJECT_ROOT)

print("Project root added:", PROJECT_ROOT)

Project root added: C:\Users\Sahil.s.Vernekar\OneDrive\Documents\ML\Churn\ml_system


In [2]:
from ml_db.mongo_client import feature_drift_reports_collection
from ml_db.mongo_client import performance_drift_reports_collection
from ml_db.mongo_client import prediction_drift_report_collection

In [3]:
feature_drift_cursor = feature_drift_reports_collection.find().sort("timestamp", -1).limit(1)
performance_drift_cursor = performance_drift_reports_collection.find().sort("timestamp", -1).limit(1)
prediction_drift_cursor = prediction_drift_report_collection.find().sort("timestamp", -1).limit(1)

feature_drift = list(feature_drift_cursor)
performance_drift = list(performance_drift_cursor)
prediction_drift = list(prediction_drift_cursor)

In [9]:
from datetime import datetime
from report.health_report_utils import overall_status, actions_data
from ml_db.mongo_client import model_health_reports_collection

report = {
  "model_version": "v1",

  "health_snapshot": {
    "feature_drift": feature_drift[0]['summary']['overall_status'],
    "prediction_drift": prediction_drift[0]['status'],
    "performance_drift": performance_drift[0]['status']
  },

  "overall_status": overall_status(performance_drift[0]['status'], prediction_drift[0]['status'], feature_drift[0]['summary']['overall_status']),

  "actions": actions_data(performance_drift[0]['status'], prediction_drift[0]['status'], feature_drift[0]['summary']['overall_status']),

  "source_reports": {
    "feature_drift_report_id": feature_drift[0]['_id'],
    "prediction_drift_report_id": prediction_drift[0]['_id'],
    "performance_drift_report_id": performance_drift[0]['_id']
  },

  "generated_at": datetime.utcnow().isoformat()
}
model_health_reports_collection.insert_one(report)

InsertOneResult(ObjectId('695619b710839dacba4edbf3'), acknowledged=True)